# Moshi Compression — Phase 1: Hidden Bootstrap Training

**Goal.** Train the student (SmolLM2-1.7B + adapters) to reproduce teacher
hidden states from cached targets.  Teacher is NOT loaded — all supervision
comes from Phase-0 cache (60 000 windows of teacher hidden + text top-256)
plus S14 Mimi codes.

**Loss.**
- Primary: cosine embedding loss on hidden states (1 - cos_sim), weight 1.0
- Secondary: cross-entropy on text logits via sparse top-256 teacher distribution, weight 0.1

**Gate to Phase 2:** validation hidden cosine similarity > 0.80 (mean 1-cos < 0.20)

**Datasets required** (attach ALL before running):
- `mhassann/moshi-cache-s{0..2}p{0..3}` (12 datasets)
- `mhassann/moshi-cache-codes`
- `tasfiatanha/moshi-frozen-heads`
- `tasfiatanha/moshi-repo`
- `tasfiatanha/moshi-compression-smoke`

**Session plan:** Each Kaggle session trains for ~8 hours, saves checkpoint,
pushes to `mhassann/moshi-p1-ckpt`. Resume from latest checkpoint next session.
Target: 30 000 steps across ~5 sessions (6 000 steps/session at ~4.5 s/step).


## Cell 1 — Global patches

In [1]:
import os, sys
# MUST be set BEFORE `import torch` or the CUDA allocator ignores it.
os.environ["TORCH_COMPILE_DISABLE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
torch._dynamo.config.disable = True
print("torch.compile disabled, expandable_segments enabled")
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))


torch.compile disabled, expandable_segments enabled
PYTORCH_CUDA_ALLOC_CONF = expandable_segments:True


## Cell 2 — Environment verification

In [2]:
print("python :", sys.version)
print("torch  :", torch.__version__, "  cuda:", torch.version.cuda)
print("device count    :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, sm {p.major}.{p.minor}, "
          f"total {p.total_memory / 1e9:.1f} GB")

assert torch.cuda.device_count() >= 1, "Need at least 1 GPU"
torch.cuda.set_device(0)
print("=== environment check PASSED ===")


python : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch  : 2.10.0+cu128   cuda: 12.8
device count    : 2
  cuda:0 = Tesla T4, sm 7.5, total 15.6 GB
  cuda:1 = Tesla T4, sm 7.5, total 15.6 GB
=== environment check PASSED ===


## Cell 3 — Installs

In [3]:
import subprocess, sys, os

for pkg in [
    "transformers==4.44.2",
    "accelerate==0.33.0",
    "bitsandbytes>=0.45.0",
    "sentencepiece",
    "einops",
]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

MOSHI_SRC = next(
    (p for p in [
        "/kaggle/input/datasets/mhassann/moshi-repo/moshi/moshi",
        "/kaggle/input/datasets/tasfiatanha/moshi-repo/moshi/moshi",
        "/kaggle/input/moshi-repo/moshi/moshi",
    ] if __import__("pathlib").Path(p).exists()),
    "/kaggle/input/datasets/tasfiatanha/moshi-repo/moshi/moshi"
)
MOSHI_DST = "/kaggle/working/moshi_repo"

if not os.path.exists(MOSHI_DST):
    ret = subprocess.run(["cp", "-r", MOSHI_SRC, MOSHI_DST], capture_output=True, text=True)
    if ret.returncode != 0:
        raise RuntimeError(f"cp failed:\n{ret.stderr}")

ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", MOSHI_DST],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print("pip stdout:", ret.stdout)
    print("pip stderr:", ret.stderr)
    raise RuntimeError("moshi editable install failed")

import site, importlib
site.addsitedir(site.getsitepackages()[0])
if MOSHI_DST not in sys.path:
    sys.path.insert(0, MOSHI_DST)
importlib.invalidate_caches()

import moshi, transformers, bitsandbytes
print(f"moshi from: {moshi.__file__}")
print(f"transformers: {transformers.__version__}")
print(f"bitsandbytes: {bitsandbytes.__version__}")
print("=== installs OK ===")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 971.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 696.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 622.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 445.0 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 1.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 909.4 kB/s eta 0:00:00
moshi from: /kaggle/working/moshi_repo/moshi/__init__.py
transformers: 4.44.2
bitsandbytes: 0.49.2
=== installs OK ===


In [4]:
import moshi.utils.compile as _moshi_compile

class _NoGraph:
    def __init__(self, fn, *a, **kw): self.fn = fn
    def __call__(self, *a, **kw):    return self.fn(*a, **kw)

_moshi_compile.CUDAGraphed = _NoGraph
print("CUDAGraphed monkey-patched to no-op")


CUDAGraphed monkey-patched to no-op


## Cell 4 — Write smol_temporal.py from S0 dataset

Copy the SmolTemporalTransformer source into the editable moshi install
so `from moshi.models.smol_temporal import SmolTemporalTransformer` works.


In [5]:
import pathlib, base64, importlib, sys, torch

DST = pathlib.Path('/kaggle/working/moshi_repo/moshi/models/smol_temporal.py')
_SRC_B64 = 'IyBtb3NoaS9tb2RlbHMvc21vbF90ZW1wb3JhbC5weQpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCmZyb20gZGF0YWNsYXNzZXMgaW1wb3J0IGRhdGFjbGFzcywgZmllbGQKZnJvbSB0eXBpbmcgaW1wb3J0IE9wdGlvbmFsCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRyYW5zZm9ybWVycwpmcm9tIC4ubW9kdWxlcy5zdHJlYW1pbmcgaW1wb3J0IFN0cmVhbWluZ01vZHVsZSwgU3RhdGUKCgpAZGF0YWNsYXNzCmNsYXNzIF9TbW9sU3RhdGUoU3RhdGUpOgogICAgcGFzdF9rZXlfdmFsdWVzOiBPcHRpb25hbFt0dXBsZV0gPSBmaWVsZChkZWZhdWx0PU5vbmUpCgogICAgZGVmIHJlc2V0KHNlbGYsIHJlc2V0X21hc2s6IHRvcmNoLlRlbnNvcikgLT4gTm9uZToKICAgICAgICBzdXBlcigpLnJlc2V0KHJlc2V0X21hc2spCiAgICAgICAgc2VsZi5wYXN0X2tleV92YWx1ZXMgPSBOb25lCgoKY2xhc3MgU21vbFRlbXBvcmFsVHJhbnNmb3JtZXIoU3RyZWFtaW5nTW9kdWxlW19TbW9sU3RhdGVdKToKICAgIGRlZiBfX2luaXRfXygKICAgICAgICBzZWxmLAogICAgICAgIHRlYWNoZXJfZGltOiBpbnQgPSA0MDk2LAogICAgICAgIHN0dWRlbnRfZGltOiBpbnQgPSAyMDQ4LAogICAgICAgIGhmX25hbWU6IHN0ciA9ICJIdWdnaW5nRmFjZVRCL1Ntb2xMTTItMS43QiIsCiAgICAgICAgcm9wZV90aGV0YTogZmxvYXQgPSAxMF8wMDAuMCwKICAgICAgICBkZXZpY2U6IHN0ciA9ICJjdWRhOjAiLAogICAgICAgIGR0eXBlOiB0b3JjaC5kdHlwZSA9IHRvcmNoLmZsb2F0MTYsCiAgICApOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYudGVhY2hlcl9kaW0gPSB0ZWFjaGVyX2RpbQogICAgICAgIHNlbGYuc3R1ZGVudF9kaW0gPSBzdHVkZW50X2RpbQoKICAgICAgICBjZmcgPSB0cmFuc2Zvcm1lcnMuQXV0b0NvbmZpZy5mcm9tX3ByZXRyYWluZWQoaGZfbmFtZSkKICAgICAgICBjZmcucm9wZV90aGV0YSA9IHJvcGVfdGhldGEKICAgICAgICBjZmcudXNlX2NhY2hlID0gVHJ1ZQogICAgICAgIGNmZy5hdHRuX2ltcGxlbWVudGF0aW9uID0gImVhZ2VyIgogICAgICAgIHNlbGYuYmFja2JvbmUgPSB0cmFuc2Zvcm1lcnMuQXV0b01vZGVsLmZyb21fcHJldHJhaW5lZCgKICAgICAgICAgICAgaGZfbmFtZSwgY29uZmlnPWNmZywgdG9yY2hfZHR5cGU9ZHR5cGUsCiAgICAgICAgKQogICAgICAgIGlmIGhhc2F0dHIoc2VsZi5iYWNrYm9uZSwgImVtYmVkX3Rva2VucyIpOgogICAgICAgICAgICBzZWxmLmJhY2tib25lLmVtYmVkX3Rva2VucyA9IG5uLklkZW50aXR5KCkKCiAgICAgICAgc2VsZi5pbl9hZGFwdGVyICA9IG5uLkxpbmVhcih0ZWFjaGVyX2RpbSwgc3R1ZGVudF9kaW0sIGJpYXM9RmFsc2UpCiAgICAgICAgc2VsZi5vdXRfYWRhcHRlciA9IG5uLkxpbmVhcihzdHVkZW50X2RpbSwgdGVhY2hlcl9kaW0sIGJpYXM9RmFsc2UpCiAgICAgICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYuaW5fYWRhcHRlci53ZWlnaHQsICBzdGQ9MS4wIC8gKHRlYWNoZXJfZGltICoqIDAuNSkpCiAgICAgICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYub3V0X2FkYXB0ZXIud2VpZ2h0LCBzdGQ9MS4wIC8gKHN0dWRlbnRfZGltICoqIDAuNSkpCgogICAgICAgIHNlbGYudG8oZGV2aWNlPWRldmljZSwgZHR5cGU9ZHR5cGUpCgogICAgZGVmIF9pbml0X3N0cmVhbWluZ19zdGF0ZShzZWxmLCBiYXRjaF9zaXplOiBpbnQpIC0+IF9TbW9sU3RhdGU6CiAgICAgICAgZGV2aWNlID0gc2VsZi5pbl9hZGFwdGVyLndlaWdodC5kZXZpY2UKICAgICAgICByZXR1cm4gX1Ntb2xTdGF0ZShiYXRjaF9zaXplPWJhdGNoX3NpemUsIGRldmljZT1kZXZpY2UsIHBhc3Rfa2V5X3ZhbHVlcz1Ob25lKQoKICAgIGRlZiBmb3J3YXJkKAogICAgICAgIHNlbGYsCiAgICAgICAgeDogdG9yY2guVGVuc29yLAogICAgICAgIGNyb3NzX2F0dGVudGlvbl9zcmM6IE9wdGlvbmFsW3RvcmNoLlRlbnNvcl0gPSBOb25lLAogICAgKSAtPiB0b3JjaC5UZW5zb3I6CiAgICAgICAgYXNzZXJ0IGNyb3NzX2F0dGVudGlvbl9zcmMgaXMgTm9uZQogICAgICAgIGFzc2VydCB4LmRpbSgpID09IDMgYW5kIHguc2hhcGVbLTFdID09IHNlbGYudGVhY2hlcl9kaW0KCiAgICAgICAgIyBNb3ZlIHRvIGJhY2tib25lIGRldmljZSAoaW5wdXQgbWF5IGNvbWUgZnJvbSBjdWRhOjEgZW1iIGxvb2t1cCkKICAgICAgICB4ID0geC50byhzZWxmLmluX2FkYXB0ZXIud2VpZ2h0LmRldmljZSkKCiAgICAgICAgcGFzdF9rdiA9IChzZWxmLl9zdHJlYW1pbmdfc3RhdGUucGFzdF9rZXlfdmFsdWVzCiAgICAgICAgICAgICAgICAgICBpZiBzZWxmLl9zdHJlYW1pbmdfc3RhdGUgaXMgbm90IE5vbmUgZWxzZSBOb25lKQoKICAgICAgICAjIEF1dG9jYXN0IHdyYXBzIEJPVEggYWRhcHRlcnMgYW5kIHRoZSBiYWNrYm9uZSBzbyBmcDMyIHRyYWluYWJsZQogICAgICAgICMgd2VpZ2h0cyBtZWV0IGZwMTYgYWN0aXZhdGlvbnMgc2FmZWx5LiBBbHNvIGtlZXBzIGdyYWRpZW50CiAgICAgICAgIyBjaGVja3BvaW50aW5nIHJlY29tcHV0YXRpb24gaW4gdGhlIHNhbWUgYXV0b2Nhc3QgY29udGV4dC4KICAgICAgICB3aXRoIHRvcmNoLmFtcC5hdXRvY2FzdCgiY3VkYSIsIGR0eXBlPXRvcmNoLmZsb2F0MTYpOgogICAgICAgICAgICBoID0gc2VsZi5pbl9hZGFwdGVyKHgpCiAgICAgICAgICAgIG91dCA9IHNlbGYuYmFja2JvbmUoCiAgICAgICAgICAgICAgICBpbnB1dHNfZW1iZWRzPWgsCiAgICAgICAgICAgICAgICBwYXN0X2tleV92YWx1ZXM9cGFzdF9rdiwKICAgICAgICAgICAgICAgIHVzZV9jYWNoZT0oc2VsZi5fc3RyZWFtaW5nX3N0YXRlIGlzIG5vdCBOb25lKSwKICAgICAgICAgICAgICAgIHJldHVybl9kaWN0PVRydWUsCiAgICAgICAgICAgICkKICAgICAgICAgICAgeSA9IHNlbGYub3V0X2FkYXB0ZXIob3V0Lmxhc3RfaGlkZGVuX3N0YXRlKQoKICAgICAgICBpZiBzZWxmLl9zdHJlYW1pbmdfc3RhdGUgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIHNlbGYuX3N0cmVhbWluZ19zdGF0ZS5wYXN0X2tleV92YWx1ZXMgPSBvdXQucGFzdF9rZXlfdmFsdWVzCgogICAgICAgIHJldHVybiB5CgogICAgZGVmIHN0dWRlbnRfc3RhdGVfZGljdChzZWxmKToKICAgICAgICByZXR1cm4gewogICAgICAgICAgICAiYmFja2JvbmUiOiAgICBzZWxmLmJhY2tib25lLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgImluX2FkYXB0ZXIiOiAgc2VsZi5pbl9hZGFwdGVyLnN0YXRlX2RpY3QoKSwKICAgICAgICAgICAgIm91dF9hZGFwdGVyIjogc2VsZi5vdXRfYWRhcHRlci5zdGF0ZV9kaWN0KCksCiAgICAgICAgfQoKICAgIGRlZiBsb2FkX3N0dWRlbnRfc3RhdGVfZGljdChzZWxmLCBzZDogZGljdCk6CiAgICAgICAgc2VsZi5iYWNrYm9uZS5sb2FkX3N0YXRlX2RpY3Qoc2RbImJhY2tib25lIl0pCiAgICAgICAgc2VsZi5pbl9hZGFwdGVyLmxvYWRfc3RhdGVfZGljdChzZFsiaW5fYWRhcHRlciJdKQogICAgICAgIHNlbGYub3V0X2FkYXB0ZXIubG9hZF9zdGF0ZV9kaWN0KHNkWyJvdXRfYWRhcHRlciJdKQo='
DST.write_bytes(base64.b64decode(_SRC_B64))
print(f'Written: {DST} ({DST.stat().st_size} bytes)')

for key in list(sys.modules.keys()):
    if 'smol_temporal' in key:
        del sys.modules[key]
importlib.invalidate_caches()

from moshi.models.smol_temporal import SmolTemporalTransformer, _SmolState
print('SmolTemporalTransformer imported OK — autocast inside forward')
print('=== Cell 4 PASSED ===')


Written: /kaggle/working/moshi_repo/moshi/models/smol_temporal.py (3539 bytes)
SmolTemporalTransformer imported OK — autocast inside forward
=== Cell 4 PASSED ===


## Cell 5 — Open cache memmap handles + codes

Open all 36 memmap handles (12 parts x 3 files) plus the codes memmap.
Define the CacheDataset that yields (codes, hidden_target, topk_idx, topk_val)
tuples for training.


In [6]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import random

WINDOWS_PER_PART = 5_000
TOTAL_WINDOWS    = 60_000
T_FRAMES         = 375
TEACHER_DIM      = 4096
N_CB             = 17
TOP_K            = 256
VAL_FRACTION     = 0.02
SEED             = 42

parts_hidden = []
parts_idx    = []
parts_val    = []

for shard in range(3):
    for part in range(4):
        for prefix in [
            f"/kaggle/input/moshi-cache-s{shard}p{part}",
            f"/kaggle/input/datasets/mhassann/moshi-cache-s{shard}p{part}",
        ]:
            import os
            if os.path.isdir(prefix):
                break
        parts_hidden.append(np.memmap(f"{prefix}/hidden.npy",
            dtype="float16", mode="r", shape=(5000, 375, 4096)))
        parts_idx.append(np.memmap(f"{prefix}/topk_idx.npy",
            dtype="int32",   mode="r", shape=(5000, 375, 256)))
        parts_val.append(np.memmap(f"{prefix}/topk_val.npy",
            dtype="float16", mode="r", shape=(5000, 375, 256)))

print(f"Opened {len(parts_hidden)} hidden + {len(parts_idx)} idx + {len(parts_val)} val memmap handles")

codes_path_candidates = [
    "/kaggle/input/moshi-cache-codes/codes.npy",
    "/kaggle/input/datasets/mhassann/moshi-cache-codes/codes.npy",
]
codes_path = None
for cp in codes_path_candidates:
    if os.path.exists(cp):
        codes_path = cp
        break
if codes_path is None:
    raise FileNotFoundError(f"codes.npy not found in: {codes_path_candidates}")

codes_mm = np.memmap(codes_path, dtype="int16", mode="r",
                     shape=(TOTAL_WINDOWS, N_CB, T_FRAMES))
print(f"Codes memmap: {codes_mm.shape} dtype={codes_mm.dtype}")

rng = random.Random(SEED)
all_indices = list(range(TOTAL_WINDOWS))
rng.shuffle(all_indices)
n_val = max(1, int(TOTAL_WINDOWS * VAL_FRACTION))
val_indices = set(all_indices[:n_val])
train_indices = [i for i in range(TOTAL_WINDOWS) if i not in val_indices]
print(f"Train: {len(train_indices)}, Val: {n_val}")


class CacheDataset(Dataset):
    def __init__(self, indices):
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        part_i, local_i = divmod(i, WINDOWS_PER_PART)

        h   = np.array(parts_hidden[part_i][local_i], copy=True)
        ti  = np.array(parts_idx[part_i][local_i], copy=True)
        tv  = np.array(parts_val[part_i][local_i], copy=True)
        c   = np.array(codes_mm[i], copy=True)

        return {
            "hidden":   torch.from_numpy(h),
            "topk_idx": torch.from_numpy(ti),
            "topk_val": torch.from_numpy(tv),
            "codes":    torch.from_numpy(c.astype(np.int64)),
        }


train_ds = CacheDataset(train_indices)
val_ds   = CacheDataset(list(val_indices))

train_loader = DataLoader(train_ds, batch_size=1, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=1, shuffle=False,
                          num_workers=0, pin_memory=True)

batch = train_ds[0]
print(f"Sample: hidden {batch['hidden'].shape} {batch['hidden'].dtype}, "
      f"codes {batch['codes'].shape} {batch['codes'].dtype}")
print("=== Cell 5 PASSED ===")


Opened 12 hidden + 12 idx + 12 val memmap handles
Codes memmap: (60000, 17, 375) dtype=int16
Train: 58800, Val: 1200
Sample: hidden torch.Size([375, 4096]) torch.float16, codes torch.Size([17, 375]) torch.int64
=== Cell 5 PASSED ===


## Cell 6 — Build student model

1. Build Moshi architecture shell (no teacher weights — `load_weight=False`)
2. Replace Helium TT with SmolTemporalTransformer
3. Load frozen heads from `tasfiatanha/moshi-frozen-heads`
4. Move everything to cuda:0
5. Cast trainable params to fp32


In [7]:
import torch, pathlib, gc
from moshi.models.loaders import CheckpointInfo
from moshi.models.smol_temporal import SmolTemporalTransformer

# ── numpy 2.x pickle compat ───────────────────────────────────────────────────
# Checkpoints saved under numpy 1.x reference numpy.core.multiarray._reconstruct
# which was moved to numpy._core in numpy 2.x.  This custom pickle_module
# intercepts find_class and redirects old paths so torch.load works without
# downgrading numpy or re-saving checkpoints.
import pickle as _pickle, io as _io

class _NumpyCompatUnpickler(_pickle.Unpickler):
    _REMAP = {
        "numpy.core.multiarray": "numpy._core.multiarray",
        "numpy.core.numeric":    "numpy._core.numeric",
        "numpy.core.umath":      "numpy._core.umath",
        "numpy.core":            "numpy._core",
    }
    def find_class(self, module, name):
        return super().find_class(self._REMAP.get(module, module), name)

class _NpPickle:
    Unpickler        = _NumpyCompatUnpickler
    loads            = staticmethod(_pickle.loads)
    load             = staticmethod(_pickle.load)
    dump             = staticmethod(_pickle.dump)
    dumps            = staticmethod(_pickle.dumps)
    HIGHEST_PROTOCOL = _pickle.HIGHEST_PROTOCOL
    DEFAULT_PROTOCOL = _pickle.DEFAULT_PROTOCOL
    PickleError      = _pickle.PickleError
    UnpicklingError  = _pickle.UnpicklingError

def _torch_load(path, **kw):
    """torch.load wrapper that handles numpy 2.x pickle compat automatically."""
    kw.setdefault("weights_only", False)
    kw.setdefault("pickle_module", _NpPickle)
    return torch.load(path, **kw)
# ─────────────────────────────────────────────────────────────────────────────


def replace_temporal_transformer(lm_model, device="cpu", dtype=torch.float16):
    teacher_dim = getattr(lm_model, "transformer_dim", 4096)
    new_tt = SmolTemporalTransformer(
        teacher_dim=teacher_dim,
        student_dim=2048,
        hf_name="HuggingFaceTB/SmolLM2-1.7B",
        rope_theta=10_000.0,
        device=device,
        dtype=dtype,
    )
    old = lm_model.transformer
    lm_model.transformer = new_tt
    del old
    torch.cuda.empty_cache()
    for name, p in lm_model.named_parameters():
        if name.startswith("transformer."):
            p.requires_grad_(True)
        else:
            p.requires_grad_(False)
    return new_tt


# Build student architecture with zero downloads.
# LMModel takes flat kwargs directly (no LMConfig wrapper in this moshi version).
# Constants from models/loaders.py in the kyutai/moshi repo.
from moshi.models.lm import LMModel

print("Building student shell from hardcoded moshiko config (no download) ...")

student_lm = LMModel(
    # ── Temporal Transformer ──────────────────────────────────────
    dim=4096,
    num_heads=32,
    num_layers=32,
    hidden_scale=4.125,        # FFN intermediate = int(4.125 * 4096) = 11264 (fused gate+up / 2)
    gating="silu",
    norm="rms_norm_f32",
    positional_embedding="rope",
    context=3000,
    # ── Codebook layout ───────────────────────────────────────────
    n_q=16,                    # 8 moshi audio + 8 user audio codebooks
    dep_q=8,                   # depformer generates 8 codebooks per step
    card=2048,                 # audio vocabulary size per codebook
    text_card=32000,           # text vocabulary size
    # ── Delays (text, moshi×8, user×8) ───────────────────────────
    delays=[0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1],
    # ── Depth Transformer ─────────────────────────────────────────
    depformer_dim=1024,
    depformer_dim_feedforward=4224,   # int(4.125 * 1024)
    depformer_num_heads=16,
    depformer_num_layers=6,
    depformer_multi_linear=True,
    depformer_weights_per_step=True,
    depformer_context=8,
    depformer_pos_emb="none",
    # ── Misc ──────────────────────────────────────────────────────
    existing_text_padding_id=3,
).to(dtype=torch.float16)

print(f"Student shell built: "
      f"{sum(p.numel() for p in student_lm.parameters())/1e9:.2f}B params on CPU")

# Swap Helium TT -> SmolLM2
print("Replacing Helium TT with SmolLM2-1.7B ...")
smol_tt = replace_temporal_transformer(student_lm, device="cpu", dtype=torch.float16)
gc.collect()
torch.cuda.empty_cache()
print(f"SmolTemporalTransformer created, backbone layers: "
      f"{smol_tt.backbone.config.num_hidden_layers}")

# Load frozen heads from dataset
FROZEN_CANDIDATES = [
    pathlib.Path("/kaggle/input/datasets/tasfiatanha/moshi-frozen-heads"),
    pathlib.Path("/kaggle/input/moshi-frozen-heads"),
]
frozen_dir = None
for p in FROZEN_CANDIDATES:
    if p.exists():
        frozen_dir = p
        break
if frozen_dir is None:
    raise FileNotFoundError(f"Frozen heads not found: {FROZEN_CANDIDATES}")

print(f"\nLoading frozen heads from {frozen_dir} ...")
frozen_modules = [
    "emb", "text_emb", "out_norm", "text_linear",
    "depformer_in", "depformer",
    "depformer_emb", "depformer_text_emb", "linears",
]
loaded = 0
for name in frozen_modules:
    pt_file = frozen_dir / f"{name}.pt"
    if pt_file.exists():
        sd = torch.load(pt_file, map_location="cpu", weights_only=True)
        getattr(student_lm, name).load_state_dict(sd)
        loaded += 1
        print(f"  loaded {name} ({sum(v.numel() for v in sd.values())/1e6:.1f}M params)")
    else:
        print(f"  MISSING {pt_file}")
print(f"Loaded {loaded}/{len(frozen_modules)} frozen modules")

# Load S0 student checkpoint if available (to resume from S0 backbone init)
S0_CKPT_CANDIDATES = [
    pathlib.Path("/kaggle/input/datasets/tasfiatanha/moshi-compression-smoke/ckpt_step_10.pt"),
    pathlib.Path("/kaggle/input/moshi-compression-smoke/ckpt_step_10.pt"),
]
for ckpt_path in S0_CKPT_CANDIDATES:
    if ckpt_path.exists():
        print(f"\nLoading S0 student weights from {ckpt_path} ...")
        s0 = _torch_load(ckpt_path, map_location="cpu")
        if "student_backbone" in s0:
            smol_tt.backbone.load_state_dict(s0["student_backbone"])
            print("  backbone loaded")
        if "in_adapter" in s0:
            smol_tt.in_adapter.load_state_dict(s0["in_adapter"])
            print("  in_adapter loaded")
        if "out_adapter" in s0:
            smol_tt.out_adapter.load_state_dict(s0["out_adapter"])
            print("  out_adapter loaded")
        break

# Memory layout (no teacher in Phase-1, both GPUs free):
#   cuda:0: student backbone + adapters (trainable, fp32 ~6.8 GB)
#           + student emb, text_emb (frozen, fp16 ~0.5 GB)
#   cuda:1: frozen heads (out_norm, text_linear, depformer* ~5.4 GB fp16)
# Total cuda:0: ~7.3 GB + grads ~6.8 GB + 8-bit optim ~1.6 GB = ~15.7 GB (tight but OK with grad ckpt)
# Total cuda:1: ~5.4 GB

# Split layout — single T4 (14.56 GB) cannot fit weights + grads + activations.
# cuda:0: emb, text_emb, transformer/SmolLM2 (trainable fp32 ~6.8 GB)
#         + grads ~6.8 GB + 8-bit optim ~1.6 GB + activations ~1 GB = ~16 GB
#         (gradient checkpointing cuts activations, expandable_segments handles fragmentation)
# cuda:1: out_norm, text_linear, depformer* (frozen fp16 ~5.4 GB)
# Forward pass: emb_sum(cuda:0) → SmolLM2(cuda:0) → out_norm pulls to cuda:1 → text_linear(cuda:1)

print("\nMoving student to split GPU layout ...")
# Trainable: transformer backbone + adapters → cuda:0
student_lm.transformer.to("cuda:0")
# ALL frozen modules → cuda:1 (saves ~1 GB on cuda:0)
# emb/text_emb output gets pulled to cuda:0 inside SmolTemporalTransformer.forward
# via: x = x.to(self.in_adapter.weight.device)
for attr in ["emb", "text_emb", "out_norm", "text_linear", "depformer_in",
             "depformer", "depformer_emb", "depformer_text_emb", "linears"]:
    if hasattr(student_lm, attr):
        getattr(student_lm, attr).to("cuda:1")

# Keep trainables in fp16. bnb.optim.PagedAdamW8bit maintains its own fp32
# master copy internally, so fp32 weight storage is redundant — and doubling 1.7B
# params on a 14.56 GB T4 is what causes OOM before a forward even runs.
# We also drop GradScaler (below) because pure fp16 weights + autocast + loss
# cast-to-fp32 is the standard T4 recipe; GradScaler.unscale_ is what required
# fp32 grads in the first place.
print("Keeping trainables in fp16 (bnb optimizer holds fp32 master copy)")

torch.cuda.synchronize()
free, total = torch.cuda.mem_get_info(0)
print(f"cuda:0: free {free/1e9:.2f} / {total/1e9:.2f} GB")

n_train = sum(p.numel() for p in student_lm.parameters() if p.requires_grad)
n_froz  = sum(p.numel() for p in student_lm.parameters() if not p.requires_grad)
print(f"\nTrainable : {n_train/1e6:.1f} M  ({n_train/1e9:.3f} B)")
print(f"Frozen    : {n_froz/1e6:.1f} M  ({n_froz/1e9:.3f} B)")
print("\n=== Cell 6 PASSED ===")

import types as _types

def _fixed_forward_text(self, sequence, sum_condition=None, cross_attention_src=None):
    """Cross-device forward_text for split layout:
      emb/text_emb on cuda:1, transformer (SmolLM2) on cuda:0,
      out_norm/text_linear on cuda:1.
    Mirrors lm.py:379-408 but routes tensors across devices.
    Autocast is INSIDE SmolTemporalTransformer.forward - do not wrap here.
    """
    B, K, S = sequence.shape
    assert K == self.num_codebooks, f"K={K} vs num_codebooks={self.num_codebooks}"

    emb_device = next(self.emb[0].parameters()).device  # cuda:1
    input_sequence = sequence.to(emb_device)

    input_ = None
    for cb_index in range(self.num_audio_codebooks):
        audio_emb = self.emb[cb_index](input_sequence[:, cb_index + self.audio_offset])
        input_ = audio_emb if input_ is None else input_ + audio_emb
    text_emb = self.text_emb(input_sequence[:, 0])
    input_ = text_emb if input_ is None else input_ + text_emb

    if sum_condition is not None:
        input_ = input_ + sum_condition.to(input_)
    if cross_attention_src is not None:
        cross_attention_src = cross_attention_src.to(input_)

    # SmolTemporalTransformer.forward does x.to(in_adapter.device) -> cuda:0,
    # runs SmolLM2 backbone under autocast(fp16), returns on cuda:0.
    transformer_out = self.transformer(input_, cross_attention_src=cross_attention_src)

    # out_norm + text_linear live on cuda:1 - move transformer_out there.
    if self.out_norm:
        on_device = next(self.out_norm.parameters()).device  # cuda:1
        transformer_out = self.out_norm(transformer_out.to(on_device))

    tl_device = next(self.text_linear.parameters()).device   # cuda:1
    text_logits = self.text_linear(transformer_out.to(tl_device))
    text_logits = text_logits[:, None]
    return transformer_out, text_logits

student_lm.forward_text = _types.MethodType(_fixed_forward_text, student_lm)
print("forward_text patched - cross-device hops before out_norm + text_linear")



Building student shell from hardcoded moshiko config (no download) ...
Student shell built: 7.69B params on CPU
Replacing Helium TT with SmolLM2-1.7B ...


config.json:   0%|          | 0.00/635 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.42G [00:00<?, ?B/s]

SmolTemporalTransformer created, backbone layers: 24

Loading frozen heads from /kaggle/input/datasets/tasfiatanha/moshi-frozen-heads ...
  loaded emb (134.3M params)
  loaded text_emb (131.1M params)
  loaded out_norm (0.0M params)
  loaded text_linear (131.1M params)
  loaded depformer_in (33.6M params)
  loaded depformer (616.6M params)
  loaded depformer_emb (14.7M params)
  loaded depformer_text_emb (32.8M params)
  loaded linears (16.8M params)
Loaded 9/9 frozen modules

Loading S0 student weights from /kaggle/input/datasets/tasfiatanha/moshi-compression-smoke/ckpt_step_10.pt ...
  backbone loaded
  in_adapter loaded
  out_adapter loaded

Moving student to split GPU layout ...
Keeping trainables in fp16 (bnb optimizer holds fp32 master copy)
cuda:0: free 12.25 / 15.64 GB

Trainable : 1627.5 M  (1.627 B)
Frozen    : 1110.8 M  (1.111 B)

=== Cell 6 PASSED ===
forward_text patched - cross-device hops before out_norm + text_linear


## Cell 7 — Optimizer, scheduler, scaler, gradient checkpointing

Phase-1 config:
- PagedAdamW8bit, lr=1e-4, weight_decay=0.01
- Linear warmup 500 steps, cosine decay to 1e-5
- GradScaler with low init_scale (2^8 to avoid early overflow)
- Gradient checkpointing on SmolLM2 backbone
- Gradient accumulation: 4 micro-batches per optimizer step


In [16]:
import bitsandbytes as bnb
import math

GRAD_ACCUM   = 4
LR           = 1e-4
LR_MIN       = 1e-5
WARMUP_STEPS = 500
MAX_STEPS    = 6_000
MAX_NORM     = 5.0   # was 1.0; raw gn ~25-30 was being over-clipped (96% signal lost)

trainable_params = [p for p in student_lm.parameters() if p.requires_grad]
assert len(trainable_params) > 0, "No trainable params!"

optimizer = bnb.optim.PagedAdamW8bit(trainable_params, lr=LR, weight_decay=0.01)
# No GradScaler: pure fp16 weights + autocast; loss reductions cast to fp32
# inside cosine_loss/sparse_text_ce (.float()). Using a no-op shim keeps Cell 11
# unchanged.
class _NoOpScaler:
    def scale(self, loss):   return loss
    def unscale_(self, opt): pass
    def step(self, opt):     opt.step()
    def update(self):        pass
    def get_scale(self):     return 1
    def state_dict(self):    return {}
    def load_state_dict(self, sd): pass
scaler = _NoOpScaler()

backbone = student_lm.transformer.backbone
if hasattr(backbone, "gradient_checkpointing_enable"):
    backbone.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False})
    print("Gradient checkpointing enabled")

def lr_schedule(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    return max(LR_MIN / LR, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)

print(f"Optimizer: PagedAdamW8bit, lr={LR}, wd=0.01")
print(f"Scheduler: linear warmup {WARMUP_STEPS} steps, cosine -> {LR_MIN}")
print(f"Grad accum: {GRAD_ACCUM}, max_norm: {MAX_NORM}")
print(f"Max steps this session: {MAX_STEPS}")
print("=== Cell 7 PASSED ===")


Gradient checkpointing enabled
Optimizer: PagedAdamW8bit, lr=0.0001, wd=0.01
Scheduler: linear warmup 500 steps, cosine -> 1e-05
Grad accum: 4, max_norm: 5.0
Max steps this session: 6000
=== Cell 7 PASSED ===


## Cell 8 — Resume from checkpoint (if exists)

Look for the latest `ckpt_step_*.pt` in the checkpoint dataset.
If found, restore student weights, optimizer, scheduler, scaler, and RNG state.


In [17]:
import pathlib, glob, json, random
import numpy as np

CKPT_DATASET_CANDIDATES = [
    "/kaggle/working",                              # in-session checkpoints first
    "/kaggle/input/datasets/mhassann/moshi-p1-ckpt",
    "/kaggle/input/moshi-p1-ckpt",
]

start_step = 0
total_wall = 0.0
ckpt_loaded = False

for ckpt_dir in CKPT_DATASET_CANDIDATES:
    if not os.path.isdir(ckpt_dir):
        continue
    ckpts = sorted(glob.glob(f"{ckpt_dir}/ckpt_step_*.pt"))
    if not ckpts:
        continue
    latest = ckpts[-1]
    print(f"Found checkpoint: {latest}")
    ckpt = _torch_load(latest, map_location="cpu")

    smol_tt.backbone.load_state_dict(ckpt["student_backbone"])
    smol_tt.in_adapter.load_state_dict(ckpt["in_adapter"])
    smol_tt.out_adapter.load_state_dict(ckpt["out_adapter"])
    print("  student weights restored")

    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    print("  optimizer + scheduler + scaler restored")

    if "torch_rng" in ckpt:
        torch.set_rng_state(ckpt["torch_rng"])
    if "cuda_rng" in ckpt:
        torch.cuda.set_rng_state_all(ckpt["cuda_rng"])
    if "numpy_rng" in ckpt:
        np.random.set_state(ckpt["numpy_rng"])
    if "python_rng" in ckpt:
        random.setstate(ckpt["python_rng"])
    print("  RNG states restored")

    start_step = ckpt.get("step", 0)
    total_wall = ckpt.get("wall_seconds", 0.0)
    ckpt_loaded = True
    del ckpt
    torch.cuda.empty_cache()
    print(f"  Resuming from step {start_step}")
    break

if not ckpt_loaded:
    print("No checkpoint found — training from scratch")

print(f"Start step: {start_step}")
print("=== Cell 8 PASSED ===")


Found checkpoint: /kaggle/working/ckpt_step_1000.pt
  student weights restored
  optimizer + scheduler + scaler restored
  RNG states restored
  Resuming from step 1000
Start step: 1000
=== Cell 8 PASSED ===


## Cell 9 — Loss functions

- **Hidden cosine loss**: 1 - cos_sim(student_hidden, teacher_hidden), averaged over (T, D)
- **Text CE from sparse top-256**: reconstruct sparse teacher distribution from
  (topk_idx, topk_val), compute KL/CE against student text logits


In [18]:
import torch
import torch.nn.functional as F

ALPHA_HIDDEN = 1.0
ALPHA_TEXT   = 0.0  # Phase 1 = hidden bootstrap. Text CE re-enabled in Phase 2.
VOCAB_SIZE   = 32000


def cosine_loss(student_h, teacher_h):
    s = student_h.float()
    t = teacher_h.float().to(s.device)  # transformer_out on cuda:0, hidden_t on cuda:1
    cos = F.cosine_similarity(s, t, dim=-1)
    return (1.0 - cos).mean()


def sparse_text_ce(student_logits, topk_idx, topk_val):
    B, T, V = student_logits.shape
    d = student_logits.device
    teacher_sparse = torch.zeros(B, T, V, device=d, dtype=torch.float32)
    teacher_sparse.scatter_(2, topk_idx.long().to(d), topk_val.float().to(d))
    teacher_probs = F.softmax(teacher_sparse, dim=-1)
    student_log_probs = F.log_softmax(student_logits.float(), dim=-1)
    return F.kl_div(student_log_probs, teacher_probs, reduction="batchmean")


print("Loss functions defined: cosine_loss, sparse_text_ce")
print(f"ALPHA_HIDDEN={ALPHA_HIDDEN}, ALPHA_TEXT={ALPHA_TEXT}")
print("=== Cell 9 PASSED ===")


Loss functions defined: cosine_loss, sparse_text_ce
ALPHA_HIDDEN=1.0, ALPHA_TEXT=0.0
=== Cell 9 PASSED ===


## Cell 10 — Validation function

Run on the held-out val set (2% of 60k = 1200 windows).
Reports mean cosine similarity — the Phase-2 gate metric.


In [19]:
@torch.no_grad()
def validate(student_lm, val_loader, device, max_batches=200):
    student_lm.eval()
    cos_sims = []
    text_losses = []

    for bi, batch in enumerate(val_loader):
        if bi >= max_batches:
            break

        codes_b    = batch["codes"].to(device)
        # hidden/logits come from cuda:1 (out_norm, text_linear live there)
        # codes → cuda:0 (emb lives there), hidden/topk → cuda:1 (text_linear lives there)
        hidden_t   = batch["hidden"].to("cuda:1")
        topk_idx_b = batch["topk_idx"].to("cuda:1")
        topk_val_b = batch["topk_val"].to("cuda:1")

        # autocast is inside SmolTemporalTransformer.forward
        transformer_out, text_logits = student_lm.forward_text(codes_b)

        cos = F.cosine_similarity(
            transformer_out.float(), hidden_t.float(), dim=-1
        ).mean().item()
        cos_sims.append(cos)

        tce = sparse_text_ce(text_logits[:, 0], topk_idx_b, topk_val_b).item()
        text_losses.append(tce)

    student_lm.train()

    mean_cos = sum(cos_sims) / max(len(cos_sims), 1)
    mean_tce = sum(text_losses) / max(len(text_losses), 1)
    return {
        "val_cos_sim": round(mean_cos, 4),
        "val_1_minus_cos": round(1.0 - mean_cos, 4),
        "val_text_ce": round(mean_tce, 4),
        "n_batches": len(cos_sims),
    }


print("validate() defined")
print("=== Cell 10 PASSED ===")


validate() defined
=== Cell 10 PASSED ===


## Cell 11 — Training loop

The main training loop. Iterates over the cache dataset with gradient
accumulation (4 micro-steps per optimizer step). Logs to `train_log.jsonl`.
Checkpoints every 1000 steps. Validates every 500 steps.


In [20]:
import time, json, pathlib

OUT_DIR = pathlib.Path("/kaggle/working")
LOG_PATH = OUT_DIR / "train_log.jsonl"

student_lm.train()
device = "cuda:0"

log_file = open(LOG_PATH, "a")

step = start_step
micro_step = 0
accum_loss_h = 0.0
accum_loss_t = 0.0
t_session = time.time()

print(f"Starting training from step {step}, max {MAX_STEPS} steps this session")
print(f"Grad accum = {GRAD_ACCUM}, effective batch size = {GRAD_ACCUM}")
print()

optimizer.zero_grad(set_to_none=True)

data_iter = iter(train_loader)

while step < start_step + MAX_STEPS:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(train_loader)
        batch = next(data_iter)

    codes_b    = batch["codes"].to(device)
    # codes → cuda:0 (emb lives there); hidden/topk → cuda:1 (text_linear lives there)
    hidden_t   = batch["hidden"].to("cuda:1")
    topk_idx_b = batch["topk_idx"].to("cuda:1")
    topk_val_b = batch["topk_val"].to("cuda:1")

    # Chunk along the time axis to cap activation memory on cuda:0.
    # T=375 runs OOM on T4; T_CHUNK=125 → 3 chunks → ~1/3 activation footprint.
    # Each chunk is an independent forward+backward; grads accumulate naturally.
    T = codes_b.shape[-1]
    T_CHUNK = 125
    micro_h = 0.0
    micro_t = 0.0
    n_chunks = (T + T_CHUNK - 1) // T_CHUNK
    for c_i in range(n_chunks):
        a, b = c_i * T_CHUNK, min((c_i + 1) * T_CHUNK, T)
        codes_c  = codes_b[..., a:b]
        hidden_c = hidden_t[:, a:b]
        idx_c    = topk_idx_b[:, a:b]
        val_c    = topk_val_b[:, a:b]

        transformer_out, text_logits = student_lm.forward_text(codes_c)
        loss_h = cosine_loss(transformer_out, hidden_c)
        loss_t = sparse_text_ce(text_logits[:, 0], idx_c, val_c)
        # Per-chunk loss is scaled by chunk_frac so the accumulated grad matches the
        # full-sequence mean (cosine_loss/kl_div are already means over their chunk).
        chunk_frac = (b - a) / T
        loss = (ALPHA_HIDDEN * loss_h + ALPHA_TEXT * loss_t) * chunk_frac / GRAD_ACCUM
        scaler.scale(loss).backward()
        micro_h += loss_h.item() * chunk_frac
        micro_t += loss_t.item() * chunk_frac

    accum_loss_h += micro_h
    accum_loss_t += micro_t
    micro_step += 1

    if micro_step % GRAD_ACCUM == 0:
        scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=MAX_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

        step += 1
        wall = time.time() - t_session + total_wall
        avg_h = accum_loss_h / GRAD_ACCUM
        avg_t = accum_loss_t / GRAD_ACCUM

        gn = float(grad_norm)
        gn_str = f"{gn:.3f}" if gn < 1e6 else "inf"

        row = {
            "step": step,
            "loss_hidden": round(avg_h, 5),
            "loss_text_ce": round(avg_t, 5),
            "grad_norm": round(gn, 4) if gn < 1e6 else None,
            "lr": round(scheduler.get_last_lr()[0], 7),
            "scale": int(scaler.get_scale()),
            "wall_s": round(wall, 1),
        }
        log_file.write(json.dumps(row) + "\n")
        log_file.flush()

        if step % 50 == 0:
            print(f"step {step:5d}  hid={avg_h:.4f}  txt={avg_t:.4f}  "
                  f"gn={gn_str}  lr={row['lr']:.2e}  "
                  f"scale={row['scale']}  wall={wall:.0f}s")

        accum_loss_h = 0.0
        accum_loss_t = 0.0

        if step % 500 == 0:
            val_result = validate(student_lm, val_loader, device)
            print(f"  VAL step {step}: cos_sim={val_result['val_cos_sim']:.4f} "
                  f"(1-cos={val_result['val_1_minus_cos']:.4f}) "
                  f"text_ce={val_result['val_text_ce']:.4f}")
            val_row = {"step": step, "type": "val", **val_result, "wall_s": round(wall, 1)}
            log_file.write(json.dumps(val_row) + "\n")
            log_file.flush()
            student_lm.train()

            if val_result["val_1_minus_cos"] < 0.20:
                print(f"  *** GATE MET: 1-cos = {val_result['val_1_minus_cos']:.4f} < 0.20 ***")
                print("  Phase-1 complete! Ready for Phase-2.")

        if step % 1000 == 0:
            ckpt_path = OUT_DIR / f"ckpt_step_{step}.pt"
            ckpt = {
                "step": step,
                "phase": "P1",
                "wall_seconds": round(wall, 1),
                "student_backbone": smol_tt.backbone.state_dict(),
                "in_adapter": smol_tt.in_adapter.state_dict(),
                "out_adapter": smol_tt.out_adapter.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "scaler": scaler.state_dict(),
                "torch_rng": torch.get_rng_state(),
                "cuda_rng": torch.cuda.get_rng_state_all(),
                "numpy_rng": np.random.get_state(),
                "python_rng": random.getstate(),
                "torch_version": torch.__version__,
            }
            torch.save(ckpt, ckpt_path)
            size_gb = ckpt_path.stat().st_size / 1e9
            print(f"  CKPT saved: {ckpt_path.name} ({size_gb:.2f} GB)")
            del ckpt

log_file.close()

final_wall = time.time() - t_session + total_wall
print(f"\nTraining done. Final step: {step}, wall: {final_wall:.0f}s")

free, total_ = torch.cuda.mem_get_info(0)
print(f"cuda:0: free {free/1e9:.2f} / {total_/1e9:.2f} GB")

val_final = validate(student_lm, val_loader, device)
print(f"Final VAL: cos_sim={val_final['val_cos_sim']:.4f} "
      f"(1-cos={val_final['val_1_minus_cos']:.4f}) "
      f"text_ce={val_final['val_text_ce']:.4f}")
print("=== Cell 11 PASSED ===")


Starting training from step 1000, max 6000 steps this session
Grad accum = 4, effective batch size = 4

step  1050  hid=0.1685  txt=53.4725  gn=0.068  lr=9.76e-05  scale=1  wall=2687s
step  1100  hid=0.1217  txt=60.1564  gn=0.045  lr=9.71e-05  scale=1  wall=2813s
step  1150  hid=0.1085  txt=62.6225  gn=0.034  lr=9.66e-05  scale=1  wall=2941s
step  1200  hid=0.1382  txt=65.2954  gn=0.038  lr=9.61e-05  scale=1  wall=3068s
step  1250  hid=0.1253  txt=65.6333  gn=0.044  lr=9.55e-05  scale=1  wall=3194s
step  1300  hid=0.1089  txt=64.5317  gn=0.033  lr=9.49e-05  scale=1  wall=3321s
step  1350  hid=0.0949  txt=63.3912  gn=0.031  lr=9.42e-05  scale=1  wall=3448s
step  1400  hid=0.1013  txt=68.2293  gn=0.032  lr=9.35e-05  scale=1  wall=3574s
step  1450  hid=0.1254  txt=67.0242  gn=0.039  lr=9.28e-05  scale=1  wall=3702s
step  1500  hid=0.1069  txt=65.7509  gn=0.042  lr=9.21e-05  scale=1  wall=3831s
  VAL step 1500: cos_sim=0.8923 (1-cos=0.1077) text_ce=199.4813
  *** GATE MET: 1-cos = 0.1077 <

KeyboardInterrupt: 

## Cell 12 — Final checkpoint + push to Kaggle

Save the final checkpoint and push it (with train_log.jsonl) to the
`mhassann/moshi-p1-ckpt` Kaggle dataset.


In [21]:
import subprocess, json, pathlib, os

OUT_DIR = pathlib.Path("/kaggle/working")
username = os.environ.get("KAGGLE_USERNAME", "mhassann")

# Save final checkpoint if not already at a 1000-step boundary
final_ckpt = OUT_DIR / f"ckpt_step_{step}.pt"
if not final_ckpt.exists():
    ckpt = {
        "step": step,
        "phase": "P1",
        "wall_seconds": round(time.time() - t_session + total_wall, 1),
        "student_backbone": smol_tt.backbone.state_dict(),
        "in_adapter": smol_tt.in_adapter.state_dict(),
        "out_adapter": smol_tt.out_adapter.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "torch_rng": torch.get_rng_state(),
        "cuda_rng": torch.cuda.get_rng_state_all(),
        "numpy_rng": np.random.get_state(),
        "python_rng": random.getstate(),
        "torch_version": torch.__version__,
    }
    torch.save(ckpt, final_ckpt)
    print(f"Final ckpt: {final_ckpt.name} ({final_ckpt.stat().st_size/1e9:.2f} GB)")
    del ckpt

# Clean up older checkpoints to save disk space for upload
ckpts = sorted(OUT_DIR.glob("ckpt_step_*.pt"))
if len(ckpts) > 1:
    for old in ckpts[:-1]:
        old.unlink()
        print(f"  deleted old: {old.name}")

# Write MANIFEST
manifest = f"""# MANIFEST - moshi-p1-ckpt

Phase 1 hidden bootstrap checkpoint.

| Key | Value |
|---|---|
| step | {step} |
| phase | P1 |
| gate_metric | val_1_minus_cos |
| gate_target | < 0.20 |
"""
(OUT_DIR / "MANIFEST.md").write_text(manifest)

# Write dataset metadata
dataset_id = f"{username}/moshi-p1-ckpt"
metadata = {
    "title":    "moshi-p1-ckpt",
    "id":       dataset_id,
    "licenses": [{"name": "CC0-1.0"}],
}
(OUT_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

# List files to upload
print("\nFiles to upload:")
upload_files = ["MANIFEST.md", "train_log.jsonl", "dataset-metadata.json"]
upload_files += [p.name for p in OUT_DIR.glob("ckpt_step_*.pt")]
total_gb = 0.0
for name in upload_files:
    p = OUT_DIR / name
    if p.exists():
        gb = p.stat().st_size / 1e9
        total_gb += gb
        print(f"  {name:<35} {gb*1000:8.1f} MB")
print(f"  {'TOTAL':<35} {total_gb:8.2f} GB")

assert total_gb < 19, f"Upload too large: {total_gb:.1f} GB"

print("\nPushing dataset ...")
r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(OUT_DIR)],
    capture_output=True, text=True,
)
print(r.stdout or "(no stdout)")
if r.returncode == 0:
    print(f"SUCCESS (create) - kaggle.com/{dataset_id}")
else:
    print(f"Create failed (rc={r.returncode}), trying version bump ...")
    r2 = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(OUT_DIR),
         "-m", f"P1 step {step}"],
        capture_output=True, text=True,
    )
    print(r2.stdout or "(no stdout)")
    if r2.returncode != 0:
        print("STDERR:", r2.stderr)
    else:
        print(f"SUCCESS (version) - kaggle.com/{dataset_id}")

print("\n=== Phase-1 session COMPLETE ===")
print(f"Step: {step}, next session resumes from this checkpoint.")


Final ckpt: ckpt_step_2058.pt (6.56 GB)
  deleted old: ckpt_step_1000.pt
  deleted old: ckpt_step_2000.pt

Files to upload:
  MANIFEST.md                              0.0 MB
  train_log.jsonl                          0.3 MB
  dataset-metadata.json                    0.0 MB
  ckpt_step_2058.pt                     6562.2 MB
  TOTAL                                   6.56 GB

Pushing dataset ...
Starting upload for file ckpt_step_2058.pt
Upload successful: ckpt_step_2058.pt (6GB)
Skipping folder: moshi_repo; use '--dir-mode' to upload folders
Starting upload for file MANIFEST.md
Upload successful: MANIFEST.md (182B)
Skipping folder: .virtual_documents; use '--dir-mode' to upload folders
Starting upload for file train_log.jsonl
Upload successful: train_log.jsonl (332KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/mhassann/moshi-p1-ckpt

SUCCESS (create) - kaggle.com/mhassann/moshi-p1-ckpt

=== Phase-1 session COMPLETE ===
Step: 2058, next 